In [0]:
# =============================================================================
# jobs/03_refresh_ea_flood_areas.py
# Weekly refresh of EA Flood Warning Areas and Flood Alert Areas.
#
# What this script does:
#   1. Fetches all Flood Warning Areas as GeoJSON from the EA geoservices API.
#   2. Fetches all Flood Alert Areas as GeoJSON from the EA geoservices API.
#   3. Derives type_code (characters 4-6 of fws_tacode) and type_text.
#   4. Overwrites ea_flood_warning_areas and ea_flood_alert_areas in full.
#
# Run schedule: weekly, Sunday 02:00 UTC.
#
# Field mapping from EA API:
#   fws_tacode -> ea_area_code
#   ta_name    -> ea_area_name
#   la_name    -> county (local authority)
#   river_sea  -> river_or_sea
#   qdial      -> quick_dial_number
#   area       -> ea_area (e.g. Wessex, Thames)
#   parent     -> parent area code
#
# type_code examples:
#   fws_tacode "111FWFBRT150" -> characters 4-6 -> "FWF" -> "Flood Warning Fluvial"
# =============================================================================

import sys
import json

sys.path.insert(0, "/Workspace/Users/jon.payne@environment-agency.gov.uk/FGS_Notebooks/")
from config import (
    EA_FWA_GEOJSON_URL, EA_FAA_GEOJSON_URL,
    TBL_EA_FWA, TBL_EA_FAA
)
from utils.helpers import get_spark, utc_now

import geopandas as gpd
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType


# =============================================================================
# SETUP
# =============================================================================

spark   = get_spark()
now_iso = utc_now().isoformat()


# =============================================================================
# TYPE CODE LOOKUP
# Characters 4-6 of fws_tacode identify the flood source and area type.
# =============================================================================

TYPE_CODE_LOOKUP = {
    "FWF": "Flood Warning Fluvial",
    "FWC": "Flood Warning Coastal",
    "FWG": "Flood Warning Groundwater",
    "FWB": "Flood Warning Both",
    "FWT": "Flood Warning Tidal",
    "WAF": "Flood Alert Fluvial",
    "WAT": "Flood Alert Tidal",
    "WAC": "Flood Alert Coastal",
    "FAG": "Flood Alert Groundwater",
    "WAB": "Flood Alert Both",
}


def extract_type_code(fwd_code: str) -> str:
    """
    Extract characters 4-6 (0-indexed: 3-6) from fws_tacode.

    Example: "111FWFBRT150" -> "FWF"

    Args:
        fwd_code: The EA area code string.

    Returns:
        Three-character type code, or empty string if the code is too short.
    """
    if fwd_code and len(fwd_code) >= 6:
        return fwd_code[3:6]
    return ""



In [0]:
# =============================================================================
# SCHEMA
# Defined explicitly to avoid type inference issues.
# =============================================================================

ea_schema = StructType([
    StructField("ea_area_code",      StringType(), True),
    StructField("ea_area_name",      StringType(), True),
    StructField("ea_area_type",      StringType(), True),
    StructField("type_code",         StringType(), True),
    StructField("type_text",         StringType(), True),
    StructField("county",            StringType(), True),
    StructField("river_or_sea",      StringType(), True),
    StructField("quick_dial_number", StringType(), True),
    StructField("parent",            StringType(), True),
    StructField("ea_area",           StringType(), True),
    StructField("geometry",          StringType(), True),
    StructField("last_refreshed_at", StringType(), True),
])


In [0]:
# =============================================================================
# FETCH AND PROCESS
# GeoPandas reads the GeoJSON FeatureCollection directly from the URL.
# =============================================================================

def fetch_and_build_rows(url: str, area_type: str, now_iso: str) -> list:
    """
    Fetch a GeoJSON FeatureCollection from the EA geoservices API,
    parse properties and geometry, derive type codes, and return Spark Rows.

    Args:
        url:       Full GeoJSON URL for the flood area layer.
        area_type: "flood_warning" or "flood_alert".
        now_iso:   Current UTC timestamp string.

    Returns:
        List of Spark Rows ready to write to Delta.
    """
    print(f"Fetching {area_type} areas from: {url}")

    gdf = gpd.read_file(url)
    print(f"  Retrieved {len(gdf)} features.")
    print(f"  Columns: {gdf.columns.tolist()}")

    rows = []

    for _, feature in gdf.iterrows():

        fwd_code  = feature.get("fws_tacode") or ""
        type_code = extract_type_code(fwd_code)
        type_text = TYPE_CODE_LOOKUP.get(type_code, "Unknown")

        # Serialise Shapely geometry back to GeoJSON string for Delta storage.
        geometry_str = json.dumps(feature.geometry.__geo_interface__) \
            if feature.geometry is not None else None

        rows.append(Row(
            ea_area_code      = fwd_code,
            ea_area_name      = str(feature.get("ta_name")   or ""),
            ea_area_type      = area_type,
            type_code         = type_code,
            type_text         = type_text,
            county            = str(feature.get("la_name")   or ""),
            river_or_sea      = str(feature.get("river_sea") or ""),
            quick_dial_number = str(feature.get("qdial")     or ""),
            parent            = str(feature.get("parent")    or ""),
            ea_area           = str(feature.get("area")      or ""),
            geometry          = geometry_str,
            last_refreshed_at = now_iso
        ))

    return rows


In [0]:
# =============================================================================
# FETCH, PARSE, AND WRITE BOTH LAYERS
# =============================================================================

for url, area_type, table_name in [
    (EA_FWA_GEOJSON_URL, "flood_warning", TBL_EA_FWA),
    (EA_FAA_GEOJSON_URL, "flood_alert",   TBL_EA_FAA),
]:
    rows = fetch_and_build_rows(url, area_type, now_iso)

    if not rows:
        raise Exception(f"No rows returned for {area_type} -- check URL and API response.")

    df = spark.createDataFrame(rows, schema=ea_schema)
    print(f"  DataFrame has {df.count()} rows.")

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    written = spark.table(table_name).count()
    print(f"Verified: {written} rows written to {table_name}.")

print("EA flood area refresh complete.")
dbutils.notebook.exit("success")

